# Implementación de Pila para Deshacer Eliminación de Pacientes

En esta mejora del proyecto ClinicLog se agregó una funcionalidad de **deshacer** o **Undo**. La idea es que, si por error se elimina un paciente, el sistema pueda recuperar su información sin tener que registrarla nuevamente de forma manual.

Para lograrlo se utilizó una estructura de datos llamada **pila**, también conocida como Stack. La pila trabaja con el principio LIFO, que significa *Last In, First Out*, o sea: el último elemento que se guarda es el primero que se puede recuperar.

En el sistema, cada vez que se elimina un paciente, antes de borrarlo se guarda en la pila toda la información necesaria para poder restaurarlo. No solo se guarda el paciente, sino también sus tratamientos y sus registros de seguimiento, ya que estos datos dependen directamente del paciente.

Por ejemplo, si se eliminan primero los datos de Germán y después los de Ana, al utilizar la opción de deshacer se recuperará primero Ana, porque fue el último paciente eliminado. Después, si se vuelve a utilizar la opción Undo, se podrá recuperar a Germán.

La pila se implementó con `collections.deque`, que es una estructura incluida en Python. Esta permite agregar elementos al final con `append()` y sacar el último con `pop()` de una forma eficiente.

También se corrigió el manejo de identificadores. Antes se generaban IDs utilizando el tiempo en milisegundos, pero esto podía ocasionar valores repetidos si varios objetos se creaban demasiado rápido. Ahora se usa un contador secuencial para asegurar que cada paciente, tratamiento y registro de seguimiento tenga un ID único.

La funcionalidad funciona de la siguiente manera:

1. El usuario selecciona un paciente desde el menú de gestión.
2. Antes de eliminarlo, se obtienen sus tratamientos y registros de seguimiento.
3. Se crea un objeto `BorradoAccion` con todos los datos eliminados.
4. El objeto se guarda en la pila mediante `push()`.
5. El paciente y sus datos relacionados se eliminan del sistema.
6. Si el usuario selecciona la opción Undo, se obtiene el último elemento de la pila con `pop()`.
7. Finalmente, se restauran el paciente, sus tratamientos y sus seguimientos.

De esta manera, la eliminación no es completamente definitiva mientras exista la acción guardada en la pila. Esto mejora la seguridad del sistema, evita perder información por accidentes y permite aplicar una estructura de datos vista en clase a una situación real dentro de una clínica.

In [1]:
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime
from typing import Deque, List, Optional


@dataclass
class Paciente:
    id: int
    nombre: str
    apellido: str
    edad: int
    condicion_medica: Optional[str] = None

    def nombre_completo(self) -> str:
        return f"{self.nombre} {self.apellido}"


@dataclass
class Tratamiento:
    id: int
    id_paciente: int
    nombre: str
    dosis: str
    frecuencia_horas: int
    costo: float


@dataclass
class RegistroSeguimiento:
    id: int
    id_paciente: int
    id_tratamiento: int
    fecha: datetime
    observaciones: str


@dataclass
class BorradoAccion:
    paciente: Paciente
    tratamientos: List[Tratamiento] = field(default_factory=list)
    seguimientos: List[RegistroSeguimiento] = field(default_factory=list)
    fecha_borrado: datetime = field(default_factory=datetime.now)


class Pila:
    def __init__(self) -> None:
        self._elementos: Deque[BorradoAccion] = deque()

    def push(self, item: BorradoAccion) -> None:
        self._elementos.append(item)

    def pop(self) -> Optional[BorradoAccion]:
        if self.is_empty():
            return None

        return self._elementos.pop()

    def peek(self) -> Optional[BorradoAccion]:
        if self.is_empty():
            return None

        return self._elementos[-1]

    def is_empty(self) -> bool:
        return len(self._elementos) == 0

    def size(self) -> int:
        return len(self._elementos)

## Clase `BorradoAccion`

Para que la pila pueda recuperar correctamente la información eliminada, se creó la clase `BorradoAccion` usando `@dataclass`.

Esta clase representa una eliminación completa dentro de ClinicLog. Cada objeto de tipo `BorradoAccion` guarda cuatro datos principales:

- El objeto `Paciente` que fue eliminado.
- La lista de tratamientos que estaban asociados a ese paciente.
- La lista de registros de seguimiento del paciente.
- La fecha y hora en la que se realizó la eliminación.

Esto es importante porque no sería suficiente guardar solamente el nombre o el ID del paciente. Si se elimina un paciente, también se eliminan sus tratamientos y seguimientos para no dejar datos sin relación dentro del sistema. Por esa razón, cuando se usa Undo, se debe restaurar toda esa información en conjunto.

El uso de `field(default_factory=list)` permite que cada acción de borrado tenga sus propias listas independientes. Así se evita que varias eliminaciones compartan accidentalmente los mismos tratamientos o seguimientos.

In [2]:
def guardar_eliminacion_en_pila(
    paciente: Paciente,
    tratamientos: List[Tratamiento],
    seguimientos: List[RegistroSeguimiento],
    pila_borrados: Pila
) -> None:
    accion_borrado = BorradoAccion(
        paciente=paciente,
        tratamientos=tratamientos.copy(),
        seguimientos=seguimientos.copy(),
        fecha_borrado=datetime.now()
    )

    pila_borrados.push(accion_borrado)

## Guardar antes de eliminar

La parte más importante de la lógica está en guardar la información antes de hacer la eliminación física.

Primero se selecciona el paciente que se desea eliminar. Después se buscan todos los tratamientos y registros de seguimiento que tengan el mismo `id_paciente`. Esa información se copia y se guarda en un objeto de tipo `BorradoAccion`.

Luego, ese objeto se agrega a la pila con el método `push()`.

Solo después de guardar la acción en la pila se eliminan los datos del sistema. Esto garantiza que exista una copia de respaldo temporal para recuperar la información si el usuario cometió un error.

La copia se hace con `.copy()` para que las listas guardadas en la pila no dependan directamente de las listas originales de los gestores.

In [3]:
def eliminar_paciente_con_undo(
    gestor_pacientes,
    gestor_tratamientos,
    gestor_seguimiento,
    paciente: Paciente,
    pila_borrados: Pila
) -> None:
    tratamientos_eliminados = gestor_tratamientos.por_paciente(paciente.id)

    seguimientos_eliminados = gestor_seguimiento.historial_paciente(
        paciente.id
    )

    accion_borrado = BorradoAccion(
        paciente=paciente,
        tratamientos=tratamientos_eliminados.copy(),
        seguimientos=seguimientos_eliminados.copy(),
        fecha_borrado=datetime.now()
    )

    pila_borrados.push(accion_borrado)

    gestor_tratamientos.eliminar_por_paciente(paciente.id)

    gestor_seguimiento.eliminar_por_paciente(paciente.id)

    gestor_pacientes.eliminar(paciente.id)

## Proceso para deshacer una eliminación

Cuando el usuario selecciona la opción **Deshacer última eliminación**, el sistema revisa primero si la pila tiene elementos.

Si la pila está vacía, significa que no hay eliminaciones guardadas y por lo tanto no hay nada que restaurar. En ese caso se muestra un mensaje informando al usuario.

Si la pila tiene datos, se utiliza el método `pop()`. Este método elimina y devuelve el último elemento almacenado en la pila. Como la pila funciona con LIFO, siempre se recupera la eliminación más reciente.

Después se vuelve a registrar el paciente en el gestor de pacientes. Luego se restauran sus tratamientos y finalmente sus seguimientos.

Antes de agregar los tratamientos y seguimientos se valida que sus IDs no existan ya dentro de los gestores. Esto ayuda a evitar que se creen registros duplicados si por alguna razón se intenta restaurar información que ya está presente.

In [4]:
def deshacer_ultima_eliminacion(
    gestor_pacientes,
    gestor_tratamientos,
    gestor_seguimiento,
    pila_borrados: Pila
) -> bool:
    if pila_borrados.is_empty():
        return False

    accion = pila_borrados.pop()

    if accion is None:
        return False

    if gestor_pacientes.obtener_por_id(accion.paciente.id) is not None:
        return False

    gestor_pacientes.registrar(accion.paciente)

    for tratamiento in accion.tratamientos:
        if not gestor_tratamientos.existe_id(tratamiento.id):
            gestor_tratamientos.registrar(tratamiento)

    for seguimiento in accion.seguimientos:
        if not gestor_seguimiento.existe_id(seguimiento.id):
            gestor_seguimiento.agregar(seguimiento)

    return True

## Corrección en la generación de IDs

Durante las pruebas se detectó un problema importante con la generación de IDs. Inicialmente se utilizaba la fecha y hora actual en milisegundos para crear los identificadores.

Aunque parece una buena idea, varios objetos pueden crearse dentro del mismo milisegundo. Por ejemplo, al inicializar los datos de prueba se crean pacientes, tratamientos y seguimientos muy rápido. Eso puede hacer que dos objetos tengan el mismo ID.

Los IDs repetidos son peligrosos porque el sistema usa esos valores para identificar qué tratamientos y seguimientos pertenecen a cada paciente. Si dos pacientes tienen el mismo ID, al eliminar uno también se pueden eliminar datos del otro.

Para evitar este problema se cambió la lógica por un contador secuencial. Cada vez que se necesita un ID, el contador aumenta en uno. Así cada objeto recibe un identificador diferente durante la ejecución del programa.

In [5]:
_contador_ids = 0


def generar_id() -> int:
    global _contador_ids

    _contador_ids += 1

    return _contador_ids

## Conclusión

Con esta mejora, ClinicLog ahora cuenta con una opción para recuperar la última eliminación realizada. La implementación utiliza una pila basada en `deque`, aplicando el principio LIFO.

La pila es adecuada para este caso porque permite recuperar primero la última eliminación, que normalmente es la que el usuario desea deshacer. Además, las operaciones principales de la pila, como agregar con `push()` y extraer con `pop()`, se realizan de manera eficiente.

También se corrigió la generación de IDs para asegurar que cada objeto tenga un identificador único. Esto evita errores en la relación entre pacientes, tratamientos y seguimientos.

Esta funcionalidad hace que el sistema sea más seguro y más cercano a una aplicación real, ya que reduce el riesgo de perder información importante por eliminar un paciente por equivocación.